In [2]:
import pandas as pd
import numpy as np

In [3]:
threads = pd.read_parquet("data/threads.parquet")
submissions = pd.read_parquet("data/submissions.parquet")

threads.info()

<class 'pandas.DataFrame'>
RangeIndex: 282497 entries, 0 to 282496
Data columns (total 9 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   comm_id                  282497 non-null  str    
 1   sub_id                   282497 non-null  str    
 2   parent_id                282497 non-null  str    
 3   subreddit_id             282497 non-null  str    
 4   author_id                282497 non-null  str    
 5   created_at               282497 non-null  str    
 6   predicted_stance_scores  167274 non-null  float64
 7   subreddit                282497 non-null  str    
 8   thread_id                282497 non-null  str    
dtypes: float64(1), str(8)
memory usage: 40.9 MB


In [4]:
threads.head(10)

,comm_id,sub_id,parent_id,subreddit_id,author_id,created_at,predicted_stance_scores,subreddit,thread_id
0,comm71631,t3_sub3423,t1_comm71632,r/4,author26899,2021-02-22 04:26:38,NaN,subreddit4,lowvyz
1,comm71633,t3_sub3423,t1_comm71634,r/4,author26899,2021-02-22 04:25:20,0.0,subreddit4,lowvyz
2,comm71635,t3_sub3423,t1_comm71636,r/4,author19952,2021-02-21 20:33:18,NaN,subreddit4,lowvyz
3,comm71634,t3_sub3423,t1_comm71637,r/4,author26900,2021-02-21 18:09:22,0.5,subreddit4,lowvyz
4,comm71636,t3_sub3423,t3_sub3423,r/4,author1577,2021-02-21 16:49:42,0.0,subreddit4,lowvyz
5,comm71638,t3_sub3423,t3_sub3423,r/4,author458,2021-02-21 16:30:23,0.0,subreddit4,lowvyz
6,comm71639,t3_sub3423,t3_sub3423,r/4,author26217,2021-02-21 16:25:18,0.0,subreddit4,lowvyz
7,comm71640,t3_sub3423,t3_sub3423,r/4,author20722,2021-02-21 16:17:39,0.0,subreddit4,lowvyz
8,comm71637,t3_sub3423,t1_comm71641,r/4,author26217,2021-02-21 16:15:13,0.0,subreddit4,lowvyz
9,comm71642,t3_sub3423,t1_comm71643,r/4,author20727,2021-02-21 15:06:49,0.5,subreddit4,lowvyz


In [5]:
from feasibility_transitions import run_feasibility
res = run_feasibility(threads, submissions)


=== checks ===
duplicate (subreddit_id, item_id) keys                      0.000
authors seen in >1 subreddit                             5278.000
users with >=2 scored items (paper evaluates 14,673)    25294.000
share of comments scored (not NI)                           0.592
share of submissions scored                                 0.801

=== score_values ===
score
0.00    58861
0.50    39829
0.90     8096
0.20     7111
0.75     7072
0.25     6734
0.10     6692
1.00     6539
0.80     6254
0.95     4517
0.85     2935
0.70     2557
0.60     2534
0.05     2322
0.30     2064

=== vs_table1 ===
              n_submissions  n_comments    paper_name  paper_comments
subreddit_id                                                         
r/0                     181        6298     r/bayarea            7743
r/1                     148        4910  r/California            7353
r/2                    2429      175033    r/collapse          194695
r/3                     385       37568        

In [6]:
cols = ["author_id", "created_at", "predicted_stance_scores"]
s = pd.concat([threads[cols], submissions[cols]])
s["score"] = pd.to_numeric(s["predicted_stance_scores"], errors="coerce")   # NI -> NaN
s = s.dropna(subset=["author_id", "score"]).copy()
s["created_at"] = pd.to_datetime(s["created_at"])

print("users with >= 1 scored item:", s["author_id"].nunique())
print("users with >= 2 scored items (any time):", (s.groupby("author_id").size() >= 2).sum())

for freq in ["1h", "6h", "12h", "1D", "7D"]:
    n_int = s.assign(bin=s["created_at"].dt.floor(freq)).groupby("author_id")["bin"].nunique()
    print(f"{freq:>3}: {(n_int >= 2).sum():>6} users in >=2 intervals | {(n_int >= 3).sum():>6} in >=3")

users with >= 1 scored item: 43861
users with >= 2 scored items (any time): 25294
 1h:  21939 users in >=2 intervals |  11617 in >=3
 6h:  19535 users in >=2 intervals |   9923 in >=3
12h:  18632 users in >=2 intervals |   9330 in >=3
 1D:  17742 users in >=2 intervals |   8836 in >=3
 7D:  15686 users in >=2 intervals |   7672 in >=3


# Phase 1
Analyse the transition probabilities of the opinions on the dataset, considering dyadic and group interactions (using Kozitsin framework)

In [7]:
from transition_tensors import fit_tensor

dy = fit_tensor(res["dyadic_events"], own="y_d", src="l_d")
gr = fit_tensor(res["group_events"][2], own="y_d", src="g_d")   # groups with >= 2 scored members

print(dy["cmi"], dy["p_perm"])   # does the source matter at all?

0.00920264628742941 0.004975124378109453


In [16]:
print(dy["cmi"], dy["cmi_null"].max(), dy["p_perm"])

0.00920264628742941 0.00023841109880615293 0.004975124378109453


In [10]:
dy["table"]

,own,src,n_events,stay,stay_95,up,up_95,down,down_95,toward
0,against,against,17910,0.604,"[0.595, 0.613]",0.396,"[0.387, 0.405]",0.000,"[0.000, 0.000]",NaN
1,against,neutral,11345,0.515,"[0.504, 0.526]",0.485,"[0.474, 0.496]",0.000,"[0.000, 0.000]",0.485
2,against,favour,14032,0.514,"[0.502, 0.525]",0.486,"[0.475, 0.498]",0.000,"[0.000, 0.000]",0.486
3,neutral,against,9068,0.290,"[0.280, 0.300]",0.199,"[0.190, 0.208]",0.511,"[0.498, 0.523]",0.511
4,neutral,neutral,6301,0.347,"[0.335, 0.359]",0.239,"[0.227, 0.249]",0.415,"[0.402, 0.428]",NaN
5,neutral,favour,7615,0.285,"[0.273, 0.296]",0.293,"[0.282, 0.305]",0.422,"[0.410, 0.434]",0.293
6,favour,against,8078,0.259,"[0.250, 0.271]",0.000,"[0.000, 0.000]",0.741,"[0.729, 0.750]",0.741
7,favour,neutral,5522,0.311,"[0.298, 0.324]",0.000,"[0.000, 0.000]",0.689,"[0.676, 0.702]",0.689
8,favour,favour,7171,0.362,"[0.350, 0.377]",0.000,"[0.000, 0.000]",0.638,"[0.623, 0.650]",NaN


In [7]:
dy_t = fit_tensor(res["dyadic_events"], "y_d", "l_d",
                  groups=["y_d", "subreddit_id", "sub_id"])

In [8]:
p = dy["N"] / dy["N"].sum()
p_sk = p.sum(1)                          # P(own, outcome)
p_s = p_sk.sum(1, keepdims=True)         # P(own)
H = -(p_sk * np.log2(p_sk / p_s)).sum()  # H(outcome | own), in bits
print(H, dy["cmi"] / H)                  # the share explained by the source

1.4919939217521236 0.006168018618080078


In [9]:
from group_fill_check import group_fill_check
out = group_fill_check(threads, submissions, res)


=== >= 2 scored members: 42270 events ===
                                out=against  out=neutral  out=favour  row_total
own     source                                                                 
against lean-against-consensus         3362         1309         924       5595
        lean-against-divided           1854          583         567       3004
        centre-consensus                536          329         266       1131
        centre-divided                 4214         1857        2069       8140
        lean-favour-consensus          1121          661         820       2602
        lean-favour-divided             491          202         336       1029
neutral lean-against-consensus         1465          714         541       2720
        lean-against-divided            788          325         284       1397
        centre-consensus                251          204         156        611
        centre-divided                 1818         1063        1210       40